# Monte Carlo Decision Analysis for Bayesian A/B Testing

This notebook demonstrates how Monte Carlo simulation can be used to support business decisions in Bayesian A/B testing.

We visualize:

- Posterior Difference Distribution
- Probability of Superiority
- Expected Loss
- Decision Boundary
- Monte Carlo Convergence

Finally, we translate statistical findings into business recommendations.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import numpy as np
import plotly.graph_objects as go

from src.BaysianABtest import BayesianABTest
from src.monte_carlo import (
    prob_B_beats_A,
    expected_uplift,
    expected_loss,
    convergence_check,
)

In [2]:
A_trials = 5000
A_success = 540

B_trials = 5000
B_success = 585

In [3]:
ab_test = BayesianABTest()

ab_test.update(
    conversions_A=A_success,
    visitors_A=A_trials,
    conversions_B=B_success,
    visitors_B=B_trials,
)

In [4]:
rng = np.random.default_rng(42)
n_samples =100000
samples_A = rng.beta(ab_test.posterior_params_A[0],
                     ab_test.posterior_params_A[1],
                     size=n_samples)
samples_B = rng.beta(
    ab_test.posterior_params_B[0],
    ab_test.posterior_params_B[1],
    size=n_samples,
                )

delta = samples_B - samples_A

In [5]:
fig = go.Figure()

fig.add_histogram(
    x=delta,
    nbinsx=80,
    histnorm="probability density",
    name="Posterior Difference",
)

fig.add_vline(
    x=0,
    line_dash="dash",
    line_color="red",
    annotation_text="No Difference",
)

fig.update_layout(
    title="Posterior Difference Distribution (θB − θA)",
    xaxis_title="Difference in Conversion Rate",
    yaxis_title="Density",
)

fig.show()

Most of the posterior mass lies to the right of zero, indicating that Variant B is more likely to have a higher conversion rate than Variant A.

The experiment provides strong evidence that Variant B outperforms Variant A. The farther the distribution is from zero, the larger the expected improvement.

Proceed to evaluate the probability of superiority and expected loss before making the final deployment decision.

In [6]:
prob_b = prob_B_beats_A(
    alpha_A=ab_test.posterior_params_A[0],beta_A=ab_test.posterior_params_A[1],
    alpha_B=ab_test.posterior_params_B[0],beta_B=ab_test.posterior_params_B[1]
)

uplift = expected_uplift(
    alpha_A=ab_test.posterior_params_A[0],beta_A=ab_test.posterior_params_A[1],
    alpha_B=ab_test.posterior_params_B[0],beta_B=ab_test.posterior_params_B[1]
)

loss = expected_loss(
    alpha_A=ab_test.posterior_params_A[0],beta_A=ab_test.posterior_params_A[1],
    alpha_B=ab_test.posterior_params_B[0],beta_B=ab_test.posterior_params_B[1]
)

prob_a = 1 - prob_b

In [7]:
print(f"P(B > A): {prob_b:.4f}")
print(f"P(A > B): {prob_a:.4f}")
print(f"Expected Uplift: {uplift:.6f}")
print(f"Expected Loss: {loss:.6f}")

P(B > A): 0.9229
P(A > B): 0.0771
Expected Uplift: 0.008991
Expected Loss: 0.000217


- Variant B has a very high posterior probability of outperforming Variant A.
- The expected uplift is positive.
- The expected loss is very small, indicating low deployment risk.


Deploying Variant B is likely to increase the conversion rate while exposing the business to minimal downside risk.
### Recommended Decision
✅ Variant B satisfies the probability and expected loss criteria. It is a strong candidate for deployment.

In [8]:
if prob_b > 0.95 and loss < 0.001:
    decision = "Deploy Variant B 🚀"
elif prob_b < 0.05:
    decision = "Keep Variant A"
else:
    decision = "Continue Experiment"

print(f"Decision: {decision}")

Decision: Continue Experiment


In [9]:
decision_table = {
    "Metric": [
        "P(B > A)",
        "Expected Loss",
        "Final Decision",
    ],
    "Value": [
        f"{prob_b:.4f}",
        f"{loss:.6f}",
        decision,
    ],
    "Threshold": [
        "> 0.95",
        "< 0.001",
        "-",
    ],
    "Status": [
        "✅" if prob_b > 0.95 else "❌",
        "✅" if loss < 0.001 else "❌",
        "-",
    ],
}

import pandas as pd

pd.DataFrame(decision_table)

,Metric,Value,Threshold,Status
0,P(B > A),0.9229,> 0.95,❌
1,Expected Loss,0.000217,< 0.001,✅
2,Final Decision,Continue Experiment,-,-


In [10]:
sample_sizes = [
    100,
    500,
    1000,
    5000,
    10000,
    25000,
    50000,
    100000,
]

probabilities = convergence_check(
            alpha_A=ab_test.posterior_params_A[0],beta_A=ab_test.posterior_params_A[1],
            alpha_B=ab_test.posterior_params_B[0],beta_B=ab_test.posterior_params_B[1],
            sample_sizes=sample_sizes,
        )


print(type(probabilities))

<class 'list'>


In [11]:
fig = go.Figure()

fig.add_scatter(
    x=sample_sizes,
    y=probabilities,
    mode="lines+markers",
    name="P(B>A)"
)

fig.update_layout(
    title="Monte Carlo Convergence",
    xaxis_title="Monte Carlo Samples",
    yaxis_title="Estimated P(B>A)",
)

fig.show()

As the number of Monte Carlo samples increases, the estimate of P(B>A) stabilizes. This demonstrates the convergence property of Monte Carlo estimation.
Larger sample sizes produce more reliable probability estimates, leading to more trustworthy business decisions.
Using 50,000–100,000 Monte Carlo samples provides a good balance between computational cost and estimation accuracy.

# Business Case Study

## Scenario

An e-commerce company is testing a redesigned checkout button.

### Variant A
- Existing design

### Variant B
- New design

After collecting user interaction data, Bayesian A/B testing was performed using Monte Carlo simulation.

## Results

- High posterior probability that Variant B outperforms Variant A.
- Positive expected uplift.
- Expected loss was evaluated before making a deployment decision.
- Decision thresholds were applied to ensure business risk remained acceptable.

## Business Recommendation

If all decision thresholds are satisfied:

- Deploy Variant B.

Otherwise:

- Continue collecting data until sufficient evidence is obtained.

This approach reduces the risk of premature deployment while enabling faster decisions than traditional fixed-sample frequentist testing.

# Conclusion

In this notebook we used Monte Carlo simulation to transform Bayesian posterior distributions into actionable business decisions.

We:

- Estimated the posterior difference distribution.
- Computed the probability that Variant B outperforms Variant A.
- Estimated expected uplift and expected loss.
- Applied Bayesian decision thresholds.
- Studied Monte Carlo convergence.
- Converted statistical evidence into business recommendations.

Together, these techniques provide a practical framework for making data-driven product decisions under uncertainty.